[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k-stacke/cmiv-ai-course/blob/master/ChatGPT_Prompting_Improved.ipynb)

# Prompting with ChatGPT

This notebook gives you hands-on experience with prompt engineering in a medical context.
It accompanies the lecture material — the goal here is to run real examples, observe
the model's behaviour, and discuss what you see with your peers.

**Sections:**
1. Setup
2. Your first medical prompt
3. Summarisation for different audiences
4. Structured data extraction from clinical text
5. Role and system prompts
6. Hallucinations — how to find and reduce them


Throughout the notebook you'll see &#9881; **Try it** boxes (suggestions to experiment) and &#128172; **Discussion** boxes (questions to talk about with your peers and the teachers).

---
## Section 0 — Setup

Run the cells below once at the start. If you are returning to this notebook after
the course, fill in your own Azure OpenAI credentials in the config cell.


In [ ]:
#%pip install --upgrade openai  # Uncomment and run if openai is not installed

import os
import json
import requests
import openai
from openai import AzureOpenAI


In [ ]:
# ── Credentials ────────────────────────────────────────────────────────────
# During the course: the URL below fetches a shared config for all participants.
# After the course:  comment out the requests block and fill in your own values.

response = requests.get('CHANGE ME')  # <-- replace with the config URL from your instructor
config = response.json()

# If you are running this after the course with your own Azure OpenAI resource,
# comment out the two lines above and uncomment the block below:
#
# config = {
#     "OPENAI_API_VERSION":    "2024-12-01-preview",           # e.g. "2024-02-01"
#     "AZURE_OPENAI_ENDPOINT": "CHANGE ME",  # <-- replace with your Azure OpenAI endpoint, e.g. "https://my-resource.openai.azure.com/"
#     "AZURE_OPENAI_API_KEY":  "CHANGE ME",  # <-- replace with your Azure OpenAI API key
#     "engine_name-chatgpt":   "gpt-4o",  # e.g. "gpt-4o"
# }


In [ ]:
# Initialise the client using the config values loaded above
client = AzureOpenAI(
    api_version=config['OPENAI_API_VERSION'],
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_key=config['AZURE_OPENAI_API_KEY'],
)

CHATGPT_ENGINE_NAME = config['engine_name-chatgpt']
print("Using engine:", CHATGPT_ENGINE_NAME)


In [ ]:
def get_completion(prompt, model=CHATGPT_ENGINE_NAME, temperature=0):
    """
    Send a single user prompt and return the model's reply as a string.

    temperature: controls output randomness.
      0   = deterministic / consistent (good for extraction and summarisation)
      0-1 = increasingly creative / varied
    """
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_with_system(prompt, system_message, model=CHATGPT_ENGINE_NAME, temperature=0):
    """
    Like get_completion, but also sends a system message.

    system_message: sets the model's persona, constraints, or context before
                    the user turn — think of it as briefing a consultant
                    before they see the patient.
    """
    messages = [
        {"role": "system",  "content": system_message},
        {"role": "user",    "content": prompt},
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


In [ ]:
# Sanity check — run this to confirm the connection works before continuing
print(get_completion("Reply with exactly the words: Connection OK"))


---
## Section 1 — Your first medical prompt

The same text, two prompts. Run both cells and compare the outputs.


In [ ]:
referral = """
Patient referred for CT abdomen. History of abdominal pain, weight loss,
and elevated CA-125. Previous ultrasound showed a 3 cm hypoechoic lesion
in the right adnexa. Patient is a 58-year-old post-menopausal woman.
"""

# Vague prompt — minimal instruction
prompt = f"Summarise this: {referral}"

print(get_completion(prompt))


In [ ]:
# Specific prompt — structured instruction with explicit output format
prompt = f"""
Summarise the clinical referral below in exactly two sentences:
1. The key clinical concern.
2. The most relevant prior finding.

Referral:
```
{referral}
```
"""

print(get_completion(prompt))


&#128172; **Discussion - about the instructions we just gave** 
* What changed between the two outputs? Which would be more useful in a real workflow, and why?
* Consider: what happens if you ask for *three* sentences instead of two?


---
## Section 2 — Summarisation for different audiences

The same CT report is summarised three ways below — for a patient, for a referring
clinician, and as a one-line triage flag. Notice how only the instruction changes;
the report text is identical in all three cells.


In [ ]:
rad_report = """CT - CT ABDOMEN AND PELVIS ENHANCED - 24-Feb-2020, 12:45 PM

INDICATION: Rule out diverticulitis.

TECHNIQUE: Volumetric image acquisition of the abdomen and pelvis with intravenous contrast.
Coronal and sagittal reformats were performed post-acquisition.

COMPARISON: CT renal colic study from 10/04/16

FINDINGS:
There are a few scattered hypoattenuating lesions in the liver likely representing cysts.
Cysts were described on the prior CT examination.

There is no evidence for colonic diverticulosis or colitis.
The appendix is identified and is normal.

There is a trace amount of pelvic free fluid identified. There is mild hazy stranding in the
left adnexal region which may be inflammatory. A definite adnexal mass or cyst is not seen
but a recently ruptured cyst could potentially demonstrate this finding. Please note the
gynecologic structures are not optimally assessed by CT.

The remainder of the intra-abdominal structures show no gross abnormality.
Lung bases are grossly clear.

No aggressive bony abnormality is identified.

OPINION:
No evidence for diverticulosis/diverticulitis/colitis. Appendix is normal.

Mild inflammatory change in the left adnexal region which may be secondary to recent
ruptured cyst. Ultrasound follow-up could be performed.
"""

In [ ]:
# Audience: patient — plain language, no jargon, max 3 short paragraphs
instruction = """
Explain the CT report below to the patient in plain, everyday language.
Avoid medical jargon. Be accurate but reassuring where appropriate.
Use no more than 3 short paragraphs.
"""

prompt = f"{instruction}\n\nReport:\n```\n{rad_report}\n```"
print(get_completion(prompt))


In [ ]:
# Audience: referring GP — specialist bullet-point summary
instruction = """
Summarise the CT report below for the referring GP.
Use bullet points under three headings:
- Key findings
- Impression
- Recommended follow-up
"""

prompt = f"{instruction}\n\nReport:\n```\n{rad_report}\n```"
print(get_completion(prompt))


In [ ]:
# Audience: ED triage — single-line flag
instruction = """
Read the CT report below. Output a single line in this exact format:
URGENT / ROUTINE / NORMAL — [one-sentence reason]
Use URGENT only if there is a finding requiring same-day clinical action.
"""

prompt = f"{instruction}\n\nReport:\n```\n{rad_report}\n```"
print(get_completion(prompt))


### Exercise

Write a prompt that summarises the CT report in a way that would be directly
useful in **your own clinical role**. Think about:
- Who is the intended reader?
- What format is most practical (bullet points, free text, table, single line)?
- What information can be omitted for your use case?


In [ ]:
# Your prompt here
instruction = """
...
"""

prompt = f"{instruction}\n\nReport:\n```\n{rad_report}\n```"
print(get_completion(prompt))


### &#9881; **Try it later if time**

Produce two summaries of the same report for two different audience levels
(e.g. patient vs. specialist, or GP vs. radiologist) in a single prompt.
Compare how the model adjusts vocabulary and level of detail.

> **Discuss:** Would you trust the patient-facing version as-is, or would it
> need review before sending? What could go wrong?


---
## Section 3 — Structured data extraction from clinical text

LLMs can extract structured fields from free-text reports — useful for populating
databases, dashboards, or downstream analysis pipelines.

The two examples below cover radiology (nodule finding) and pathology (breast biopsy).


In [ ]:
# Radiology: extract nodule data from a free-text report
instruction = """
Extract information about all nodules found in the report below.
Return the result as a JSON list. Each nodule should be an object with fields:
  - size (in cm)
  - location
If no nodules are found, return an empty list [].
"""

radio_text = "One nodule found, 2.5 x 2.5 cm, located in the lower-left part of the liver."

prompt = f"{instruction}\n\nReport:\n```\n{radio_text}\n```"
print(get_completion(prompt))


In [ ]:
path_report = """
Specimen: Core needle biopsy, right breast, 2:00 position.
Diagnosis: Invasive ductal carcinoma, grade 2 (Nottingham score 6/9).
Tumour size: 1.8 cm (radiological). Surgical margins: Free (closest margin 3 mm, posterior).
Hormone receptors: ER positive (Allred score 7/8), PR positive (Allred score 5/8).
HER2: Negative (IHC score 1+). Ki-67 proliferation index: 18%.
Lymphovascular invasion: Not identified.
Ductal carcinoma in situ (DCIS): Present, intermediate grade.
"""

# Pathology: extract structured fields from a breast biopsy report
instruction = """
Extract the following fields from the pathology report below and return them as JSON.
If a field is not mentioned in the report, use null.

Fields:
  - tumour_type
  - grade
  - tumour_size_cm
  - er_status
  - pr_status
  - her2_status
  - ki67_percent
  - lymphovascular_invasion (true / false / null)
  - margins_clear (true / false / null)
"""

prompt = f"{instruction}\n\nReport:\n```\n{path_report}\n```"
print(get_completion(prompt))


In [ ]:
# What happens when you ask for a field that is not in the report?
instruction = """
Extract the following fields from the pathology report below and return them as JSON.
If a field is not mentioned in the report, use null.

Fields:
  - tumour_type
  - grade
  - patient_age
  - smoking_status
  - tumour_size_cm
"""

prompt = f"{instruction}\n\nReport:\n```\n{path_report}\n```"
print(get_completion(prompt))


> **Observe:** Does the model return `null` for the absent fields, or does it
> attempt to infer or guess them? This is relevant to the hallucination discussion
> in Section 6.


### Exercise

Choose one of the two reports above (or write your own short synthetic report)
and define a new set of fields to extract — ones that would be useful in your
clinical or research workflow.

Self-check: Does the output contain any values that are not actually in the report text?


In [ ]:
# Your extraction prompt here
instruction = """
...
"""

your_text = """
...
"""

prompt = f"{instruction}\n\nReport:\n```\n{your_text}\n```"
print(get_completion(prompt))


### &#9881; **Try if time**

Modify your prompt to return the output in a different format: a Python dict,
a CSV row, or a Markdown table. Does the model handle all formats equally well?

> **Discuss:** If you were building an automated pipeline that processes 500 reports
> per day, what additional validation would you put around this model output
> before it reaches a database?


---
## Section 4 — Role and system prompts

A **system message** is sent before the user turn and shapes how the model behaves
throughout the conversation — its persona, expertise level, and constraints.
Think of it as the briefing you give a specialist before they see the patient.

The `get_completion_with_system()` function defined in Setup accepts a `system_message`
argument alongside the prompt.


In [ ]:
# The same clinical question, without a system prompt
question = "What does a Ki-67 of 18% mean for a breast cancer patient's prognosis?"

print("--- No system prompt ---")
print(get_completion(question))


In [ ]:
# With a specialist system prompt — notice the difference in tone and depth
system = """
You are a clinical decision support assistant for oncologists.
Answer concisely and at a specialist level.
Reference established clinical guidelines where relevant.
If a question falls outside your knowledge, say so explicitly.
"""

print("--- With specialist system prompt ---")
print(get_completion_with_system(question, system))


In [ ]:
# System prompt as a safety constraint:
# force the model to stay within the provided text
system = """
You are a medical report assistant.
Answer questions ONLY using information explicitly present in the provided report text.
If the answer is not stated in the text, respond with exactly: "Not stated in the report."
Do not use outside knowledge.
"""

question_in_report = "Was diverticulitis found?"
question_not_in_report = "What was the patient's haemoglobin level?"

print("--- Question answered by the report ---")
print(get_completion_with_system(f"{question_in_report}\n\nReport:\n{rad_report}", system))

print("\n--- Question NOT in the report ---")
print(get_completion_with_system(f"{question_not_in_report}\n\nReport:\n{rad_report}", system))


### Exercise

Write a system prompt for a specific clinical workflow from your own specialty.
Examples:
- A radiology second-opinion assistant that flags incidental findings
- A pathology assistant that always reports fields in a fixed template
- A triage support tool that only escalates findings above a defined threshold

Then ask it 2–3 questions and check that it stays within the role you defined.


In [ ]:
# Your system prompt
system = """
...
"""

# Your question(s)
question = "..."

print(get_completion_with_system(question, system))


### &#9881; **Try if time**

Try to **break** your system prompt: find a query that makes the model step
outside the constraints you set. Then update the system prompt to prevent it.

> **Discuss:** What kinds of constraints are easy to enforce via system prompts?
> What kinds are harder? What does this imply for deploying a clinical AI tool in production?


---
## Section 5 — Hallucinations: how to find and reduce them

LLMs have no internal model of truth. They generate plausible-sounding text based
on patterns — which means they can produce confident, fluent, and **completely wrong**
outputs. This is called hallucination, and it is the most important failure mode
to understand for any clinical application.

The cells below demonstrate two common hallucination traps.


In [ ]:
# Trap 1: asking for specific peer-reviewed citations.
#
# Note: in this notebook we call the model directly via the API — it has NO
# internet access and NO retrieval tools. Anything that looks like a citation
# in the output is produced from training data alone, by pattern, not by lookup.

instruction = """
Provide three peer-reviewed studies published between 2017 and 2022 that
validated the Fleischner Society 2017 recommendation of a 6 mm threshold
for routine follow-up of incidental solid pulmonary nodules in low-risk
adults on chest CT.

For each study, give:
  - First author and co-authors
  - Title
  - Journal, year, volume, pages
  - PMID
"""

print(get_completion(instruction))


> **Observe:** The model produced three confidently-formatted citations — authors, journals, page numbers, PMIDs. Pick one and check it on PubMed.
>
> Many ChatGPT and Copilot interfaces *do* now have web search built in, and there a request like this would trigger a retrieval step. But the underlying language model on its own cannot look anything up. Whenever you use an LLM through an API, through a custom integration, or in any setting where retrieval is not explicitly wired up, you are seeing this raw behaviour — and even retrieval-augmented systems can fall back to fabrication when a search returns nothing relevant.

---

The next trap is more subtle. Citations are obvious to spot once you check them — but the same fabrication mechanism is at work whenever the model is asked to produce structured output where some of the required input is missing. The output looks correct, uses the right framework, and quietly invents the gaps.


In [ ]:
# Trap 2 (radiology): applying a real classification system to a report that
# is MISSING some of the required inputs.
#
# Lung-RADS v2022 categories depend on:
#   - the exam being a lung cancer screening LDCT in an eligible patient
#   - nodule attenuation (solid / part-solid / pure GGN)
#   - comparison with a prior screening study (new vs. stable vs. growing)
#
# The CT report below describes an incidentally detected nodule on a
# diagnostic chest CT, with no prior study available and no screening
# context. A Lung-RADS category therefore cannot be assigned — but watch
# what the model does anyway.

ct_report = """
CT CHEST WITH CONTRAST — 14-Mar-2024

INDICATION: Persistent cough, 62-year-old patient.

COMPARISON: None available.

FINDINGS:
A 7 mm nodule is identified in the right upper lobe, posterior segment.
The remainder of the lung parenchyma is clear. No pleural effusion.
Mediastinal and hilar lymph nodes are within normal size limits.
No osseous lesions.

IMPRESSION:
7 mm right upper lobe pulmonary nodule. Recommend correlation with any
prior imaging and clinical follow-up as appropriate.
"""

instruction = """
Assign the Lung-RADS v2022 category for the pulmonary nodule described
in the CT report below. State the category and the recommended follow-up
interval.
"""

prompt = f"{instruction}\n\nReport:\n```\n{ct_report}\n```"
print(get_completion(prompt))

> **Observe:** This output is far harder to catch than the fabricated citations from Trap 1. The model used the correct framework (Lung-RADS v2022), a plausible category, and a realistic follow-up interval. Spotting the problem requires knowing (1) that Lung-RADS applies **only to lung cancer screening LDCTs in eligible patients**, not to diagnostic chest CTs ordered for symptoms, (2) that a category depends on nodule attenuation (solid / part-solid / pure GGN), which the report does not state, and (3) that growth assessment requires a prior study, which the report explicitly says is unavailable. The model has silently substituted "solid attenuation" and a "baseline screen" interpretation, often justifying the category from the 7 mm size alone — a size cutoff that only carries meaning once the other inputs are fixed.
>
> This is the dangerous category of hallucination: fluent, structurally correct, quietly wrong — and only visible to someone with the domain knowledge to know what *should* have been required as input.

In [ ]:
# Mitigation: explicitly forbid the model from filling in missing data.

instruction = """
Assign the Lung-RADS v2022 category for the pulmonary nodule described
in the CT report below.

Rules:
- Use only information explicitly stated in the report.
- If any input required by Lung-RADS is missing (e.g. screening context,
  nodule attenuation, comparison with a prior study), state which
  information is missing and do NOT assign a category.
- Do not assume default values such as "solid nodule", "no prior growth",
  or "screening-eligible patient" when these are not stated.
"""

prompt = f"{instruction}\n\nReport:\n```\n{ct_report}\n```"
print(get_completion(prompt))

### Exercise 

Design a prompt that is **deliberately intended** to make the model hallucinate.
Useful strategies:
- Ask for a guideline, grading system, or classification from a specific
  professional body (national guidelines, institution-specific protocols)
- Ask it to retrieve a measurement or value not present in the report
- Ask it to compare two reports when only one is provided

Then modify the prompt to **prevent** the hallucination.
Self-check: does your mitigation work for different texts, or just this one?


In [ ]:
# Step 1: your hallucination-inducing prompt
prompt_trap = """
...
"""
print("--- Trap ---")
print(get_completion(prompt_trap))


In [ ]:
# Step 2: your mitigated prompt
prompt_safe = """
...
"""
print("--- Mitigated ---")
print(get_completion(prompt_safe))


### &#9881; **Try if time**

Add a `system_message` (from Section 4) to your mitigated prompt.
Does combining a safety-focused system prompt with a careful user prompt
give better protection than either alone?

> **Discuss:** In your department's workflow, at what step would a hallucinated
> output most likely be caught — or might it not be caught at all?
> What does this imply for where human review must sit in an AI-assisted pipeline?
